# Part 2: Vocabulary, Naive Bayes, and Log-Ratios
Each review is treated as a **bag of unigrams and bigrams** (word order is ignored).

### 0. Setup (from Part 1)
Imports, the balanced IMDb subset, `tokenize()` and `calculate_metrics()` — identical to Part 1 so this notebook runs on its own.

In [1]:
import re
import html
import numpy as np
from collections import Counter
from datasets import load_dataset

imdb = load_dataset("stanfordnlp/imdb")

def extract_balanced_subset(dataset, samples_per_class, seed=42):
    texts, labels = [], []
    for label in [0, 1]:
        class_indices = [i for i, item in enumerate(dataset) if item["label"] == label]
        rng = np.random.default_rng(seed + label)
        for index in rng.choice(class_indices, size=samples_per_class, replace=False):
            texts.append(dataset[int(index)]["text"])
            labels.append(label)
    order = np.random.default_rng(seed).permutation(len(texts))
    return [texts[i] for i in order], np.array([labels[i] for i in order])

train_texts, y_train = extract_balanced_subset(imdb["train"], 2500, seed=42)
test_texts, y_test = extract_balanced_subset(imdb["test"], 500, seed=123)

def clean_text(text):
    text = html.unescape(text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = text.lower()
    return re.sub(r"\s+", " ", text).strip()

def tokenize(text):
    tokens = re.findall(r"[a-z]+(?:'[a-z]+)?", clean_text(text))
    return ["<s>"] + tokens + ["</s>"]

def calculate_metrics(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    accuracy = (tp + tn) / len(y_true)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"Accuracy": accuracy, "Precision": precision, "Recall": recall, "F1-Score": f1}

print("Train:", len(train_texts), "| Test:", len(test_texts))

C:\Users\Rana\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\Rana\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Rana\.cache\huggingface\hub\datasets--stanfordnlp--imdb. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


C:\Users\Rana\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Rana\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating train split:  20%|██        | 5000/25000 [00:00<00:00, 40419.94 examples/s]

...

Generating unsupervised split:  24%|██▍       | 12000/50000 [00:00<00:00, 98801.87 examples/s]

Generating unsupervised split:  52%|█████▏    | 26000/50000 [00:00<00:00, 117276.33 examples/s]

Generating unsupervised split:  82%|████████▏ | 41000/50000 [00:00<00:00, 127734.86 examples/s]

Generating unsupervised split: 100%|██████████| 50000/50000 [00:00<00:00, 125750.40 examples/s]

Train: 5000 | Test: 1000


### 1. Feature extraction
We re-tokenize the raw texts with `tokenize()` from Part 1 (Part 1's token lists contain `<UNK>`), drop the `<s>` / `</s>` padding, and count every unigram and bigram in each review.

In [2]:
def extract_features(text):
    words = tokenize(text)[1:-1]          # remove <s> and </s>
    features = Counter(words)             # unigrams
    for i in range(len(words) - 1):       # bigrams
        features[words[i] + " " + words[i + 1]] += 1
    return features

train_feature_counts = [extract_features(t) for t in train_texts]
test_feature_counts  = [extract_features(t) for t in test_texts]

print("Example:", dict(extract_features("Not good, not good at all.")))
print("Train reviews:", len(train_feature_counts), "| Test reviews:", len(test_feature_counts))

Example: {'not': 2, 'good': 2, 'at': 1, 'all': 1, 'not good': 2, 'good not': 1, 'good at': 1, 'at all': 1}
Train reviews: 5000 | Test reviews: 1000


### 2. Vocabulary
The 2,000 most frequent unigrams and bigrams combined, counted on the **training set only**. Each feature is mapped to a unique index; everything else is ignored.

In [3]:
total_counts = Counter()
for features in train_feature_counts:
    total_counts.update(features)

top_features = [f for f, _ in total_counts.most_common(2000)]
feature_to_index = {f: i for i, f in enumerate(top_features)}
V = len(feature_to_index)

print("Vocabulary size:", V)
print("Bigrams in vocabulary:", sum(" " in f for f in top_features))
print("Top 15 features:", top_features[:15])

Vocabulary size: 2000
Bigrams in vocabulary: 842
Top 15 features: ['the', 'and', 'a', 'of', 'to', 'is', 'in', 'it', 'i', 'this', 'that', 'was', 'as', 'for', 'with']


### 3. Count matrices
$C_{i,j}$ = count of feature $j$ in review $i$, shape `(num_reviews, 2000)`.

In [4]:
def build_count_matrix(feature_counts):
    C = np.zeros((len(feature_counts), V))
    for i, features in enumerate(feature_counts):
        for f, count in features.items():
            if f in feature_to_index:
                C[i, feature_to_index[f]] = count
    return C

C_train = build_count_matrix(train_feature_counts)
C_test  = build_count_matrix(test_feature_counts)

print("C_train:", C_train.shape, "| C_test:", C_test.shape)

C_train: (5000, 2000) | C_test: (1000, 2000)


### 4. Probabilities with Add-1 smoothing
$$P(f_j \mid c) = \frac{\text{count}(f_j, c) + 1}{\sum_k \text{count}(f_k, c) + |V|}$$

In [5]:
pos_counts = C_train[y_train == 1].sum(axis=0)
neg_counts = C_train[y_train == 0].sum(axis=0)

p_pos = (pos_counts + 1) / (pos_counts.sum() + V)
p_neg = (neg_counts + 1) / (neg_counts.sum() + V)

print(f"Sum P(f|Pos) = {p_pos.sum():.4f} | Sum P(f|Neg) = {p_neg.sum():.4f}")

Sum P(f|Pos) = 1.0000 | Sum P(f|Neg) = 1.0000


### 5. Log-ratios
$$r_j = \log\frac{P(f_j \mid \text{Pos})}{P(f_j \mid \text{Neg})}$$
$r_j > 0$ leans positive, $r_j < 0$ leans negative, $r_j \approx 0$ is neutral.

In [6]:
r = np.log(p_pos) - np.log(p_neg)

order = np.argsort(r)
print("Most positive:", [top_features[j] for j in order[::-1][:15]])
print("Most negative:", [top_features[j] for j in order[:15]])

Most positive: ['a must', 'must see', 'fantastic', 'favorite', 'an excellent', 'wonderful', 'brilliant', 'powerful', 'my favorite', 'excellent', 'perfectly', 'incredible', 'amazing', 'superb', 'the greatest']
Most negative: ['the worst', 'waste of', 'waste', 'awful', 'lame', 'worst', 'so bad', 'poorly', 'wasted', 'dull', 'worse', 'crap', 'avoid', 'zombie', 'terrible']


### 6. Naive Bayes prediction and evaluation
$$\text{score}(d) = \log\frac{P(\text{Pos})}{P(\text{Neg})} + \sum_j \text{count}(f_j, d)\cdot r_j$$
Predict positive if score > 0. The prior ratio is 0 here because the training set is balanced. Metrics use `calculate_metrics()` from Part 1.

In [7]:
log_prior = np.log(np.sum(y_train == 1) / np.sum(y_train == 0))

def predict_nb(C):
    return (C @ r + log_prior > 0).astype(int)

y_pred_nb = predict_nb(C_test)
nb_metrics = calculate_metrics(y_test, y_pred_nb)

print("Log prior ratio:", log_prior)
print("\nNaive Bayes Results")
for metric, value in nb_metrics.items():
    print(f"{metric}: {value:.4f}")

Log prior ratio: 0.0

Naive Bayes Results
Accuracy: 0.8040
Precision: 0.8128
Recall: 0.7900
F1-Score: 0.8012


### 7. Conclusions

- **Performance:** Naive Bayes with a 2,000-feature unigram + bigram vocabulary reaches **80.4% accuracy** (F1 = 0.80) on 1,000 unseen test reviews.
- **Balanced errors:** Precision (0.81) and recall (0.79) are close, so the model is not strongly biased toward either class. It misses slightly more positive reviews than it mislabels negative ones.
- **The model learned real sentiment:** The log-ratios put clear praise words (`excellent`, `superb`, `must see`) at the positive end and clear criticism (`the worst`, `waste of`, `awful`) at the negative end, while frequent function words like `the` stay near 0.
- **Bigrams help:** 842 of the 2,000 features are bigrams, and several of the strongest features (`a must`, `must see`, `waste of`, `so bad`) are bigrams. They capture phrases that single words miss.
- **Dataset bias:** Some features reflect the data rather than sentiment. For example, `zombie` ranks as strongly negative because zombie movies in IMDb tend to get poor reviews.
- **Limitations:** The model ignores word order beyond adjacent pairs, assumes features are independent, and drops everything outside the top-2,000 vocabulary. Negation over longer spans (e.g. "not at all good") and sarcasm are not handled.
- **No prior effect:** The training set is balanced, so the log prior is 0 and predictions depend only on the review's features.